<h1 align="center"><b>AmberTools Tutorials with T4 Lysozyme Enzyme (3HTB)</b></h1>


<div align="center">
  <img src="https://github.com/SHEDOOMTC/Basic_Moco_2/raw/main/Assets/Moco_2_visuals.png" alt="Description" width="400"/>
</div>

<h2 style="text-align: left; color: darkred; font-weight: bold;">About and Usage</h2>

This tutorial will take us through all the steps of **molecular dynamics simulations** in **[AmberTools](https://ambermd.org/AmberTools.php)**, from **system preparation** to **Production run**.

This is a practical session follow up of the course on **"[Basic Molecular Modelling Course for Chemists and Life Sciences II](https://github.com/SHEDOOMTC/Basic_Moco_2)** ***(Basic Moco II)***"

Cite this tutorial as **Eze, S., Nnemolisa, S., Munachi, E., Ignatius, S., Oluyemi, W., & Adeniyi, A. (2026, YYYYY). SHEDOOMTC/Basic_Moco_2: v.X.X.X. Zenodo. https://doi.XXXX**

You can run this tutorial online without installing anything or you can download the files and run it locally on your system. To run it online, click on this **[Google Colab notebook]()**


In [ ]:
#If you want to skip the whole simulation, then copy the full simulation from ../AmberTools_run_done 
#!cp -r ../AmberTools_run_done/* .

In [ ]:
#Import Libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import nglview as nv

<h3 style="text-align: left; color: darkblue; font-weight: bold;">Introducing our Protein</h3>

 **[3HTB](https://www.rcsb.org/structure/3HTB)** (T4 lysozyme) is a premier benchmark structure in computational chemistry, representing a mutant of T4 Lysozyme complexed with an engineered binder. It is considered one of the ultimate systems for learning or evaluating molecular dynamics (MD) because it features a perfectly enclosed, buried ligand cavity.

 3HTB is complexed with 2-propylphenol, the co-crystal ligand

- **Download the protein, and inspect it**

In [ ]:
%cd /home/shadrach/Basic_Moco_2/AmberTools_run

In [ ]:
!wget "https://files.rcsb.org/download/3HTB.pdb"

!grep "^ATOM" 3HTB.pdb | wc
!grep "^SEQRES" 3HTB.pdb | wc
!grep "^HETATM" 3HTB.pdb

<font color="darkred">*How many atoms does the protein have?* *How many residues does it have?* *How many different hetatoms are present?*</font>

- **Remove water and ions and extract the ligand**

We have many co-crystal ligands and our ligand of interest is **JZ4**. So, we will extract the protein separately and the ligand separately.
The ligand belongs to chain A

In [ ]:
!grep "^ATOM" 3HTB.pdb > 3HTB_protein.pdb
!grep "^HETATM" 3HTB.pdb | grep "JZ4 A" > jz4_ligand.pdb

In [ ]:
!cat jz4_ligand.pdb

- **Visualize and inspect the protein and ligand**

In [ ]:
view = nv.show_file("3HTB_protein.pdb")
view

In [ ]:
view = nv.show_file("jz4_ligand.pdb")
view

<h3 style="text-align: left; color: darkblue; font-weight: bold;">Prepare the protein and ligand for MD simulations</h3>

Proteins from crystallographic database ussually lack all that is necessary fro a hitch-free simulation. X-ray structures lack hydrogens while some have missing residues. During protein preparation, we correct all these defaults.

In addition, we assign forcefield and atomtype parameters to our ligands and proteins

**[Antechamber](https://ambermd.org/antechamber/antechamber.pdf)** is a set of programs for molecular mechnanics; recognizing the atom type; recognizing bond type; judging the atomic equivalence; generating residue topology file; finding missing force field parameters; supplying reasonable and similar substitutes.

**[PDB4amber]()** is used for protein preparation; adds missing sections, disulfide bonds and different HIS protonation states; useful prep for constant pH simulations; pdb4amber analyses and cleans PDB files for further usage

- **Ligand Preparation**

In [ ]:
#Create a folder for system preparations and enter into it
!mkdir system_prep
%cd system_prep

#copy  your protein and ligand to the folder
!cp ../3HTB_protein.pdb .
!cp ../jz4_ligand.pdb .

In [ ]:
#If you want to skip the whole system preparation step, just copy an already solved prepared system from ./system_prep_done
#!cp ../../AmberTools_run_done/system_prep_done/* .

In [ ]:
#Prepare the ligand
!antechamber -fi pdb -fo mol2 -i jz4_ligand.pdb -o jz4_ligand.mol2 -c bcc -j 4 -at gaff

<font color="darkred">What other options are available for the **forcefield (-at flag)** and **charge models (-c flag)** in amber parametrization? See **"antechamber -h"**</font>


In [ ]:
#!antechamber -h

In [ ]:
#To ensure all the atoms are correctly typed, and parameters assigned, we run parmchk2 to generate a frcmod file for the ligand
!parmchk2 -i jz4_ligand.mol2 -o jz4_ligand.frcmod -f mol2 -s gaff2

#visualize the jz4_ligand.frcmod file; what parameters were added? what parameters were missing? what parameters were changed?
%cat jz4_ligand.frcmod

<font color="darkred">Is there a program that wraps all these steps into one and parametrizes the files as needed in a single command, for small molecules? Meet **[ACPYPE]()**</font>

**[ACPYPE (AnteChamber PYthon Parser interfacE)](https://github.com/alanwilter/acpype)** is a lightweight Python tool that generates topology and parameter files for molecular dynamics simulations. It acts as a wrapper around ANTECHAMBER, simplifying the preparation of small molecules for use with GROMACS, AMBER, and other MD packages, ensuring compatibility and efficiency. It wraps around antechamber, parmchk2 and tleap for small molecules.

- **Protein preparation**

In [ ]:
#Prepare the protein
!pdb4amber -i 3HTB_protein.pdb -o 3HTB_protein_prep.pdb  --reduce --dry --add-missing-atoms  --logfile 3HTB_prep.log

#reduce is to add hydrogens; add-missing-atoms help fill in missing residues and obviously we keep the run in a log file

<font color="darkred">Why do we apply **"reduce" flag**? What is the importance of hydrogens in dynamics simulations? Does it affect our computational cost</font>

In [ ]:
#Visualize the prepared protein
view = nv.show_file("3HTB_protein.pdb")
view

<h3 style="text-align: left; color: darkblue; font-weight: bold;">Build your Solvated System for MD simulations</h3>


**[TLEAP]()** is the engine used to construct the full simulation setup; ollates parameters to make a complete description of a molecule; solvates the system in a box of solvents (water); add ions to a specific concentration or to neutrality; generates topologies and coordinate files as outputs

- **Run the TLeap program**

In [ ]:
%%bash
#Create a tleap input file to generate the topology and coordinate files for the protein-ligand complex
cat << 'END' > tleap.in
source leaprc.protein.ff14SB # Load the protein force field
source leaprc.gaff2 # Load the ligand force field
source leaprc.water.tip3p # Load the water model
loadAmberParams jz4_ligand.frcmod
set default PBradii mbondi3 # Set the default PBradii value
receptor = loadpdb 3HTB_protein_prep.pdb # Load the protein structure
jz4 = loadmol2 jz4_ligand.mol2 # Load the ligand structure
complex = combine {receptor jz4} # Combine the receptor and ligand
saveAmberParm jz4 jz4.top jz4.crd # Save the ligand topology and coordinates
saveAmberParm receptor rec.top rec.crd # Save the receptor topology and coordinates
saveAmberParm complex com.top com.crd # Save the complex topology and coordinates	
savepdb complex complex_gas.pdb # Save the complex structure in PDB format		
solvateBox complex TIP3PBOX 12.0 # Solvate the complex in a water box with a 12.0 Å buffer
addIons complex Na+ 0 # Add sodium ions
addIons complex Cl- 0 # Add chloride ions
saveAmberParm complex complex.top complex.crd #Save the solvated, neutralized complex
savePdb complex complex.pdb
quit
END

#visualize the tleap input file
cat tleap.in

<font color="darkred">Why did we use the TIP3P water model, what other water models are available?</font>

<font color="darkred">We used cubic box, could it have been better to use octahedron box (solvateOct)?</font>

In [ ]:
!tleap -f tleap.in > tleap.log

In [ ]:
!ls

- **Visualize the solvated complex**

In [ ]:
view = nv.show_file("complex.pdb")
view.clear_representations()
view.add_representation("cartoon", selection="protein")
view.add_representation("line", selection="JZ4")
#view.add_representation("line", selection="WAT") 
view.add_representation("sphere", selection="ions")
view

- **Inspect your topology file**

Print out all the key headers in your topology file

All description of atomtypes and parameters are stored at each "FLAG" header which can be retrieved by running the code below

In [ ]:
!grep "^%FLAG" complex.top

<font color="darkred">What is a topology file and what role does it play in MD simulation? How is topology files different from coordinate files?</font>

<h3 style="text-align: left; color: darkblue; font-weight: bold;">Energy Minimization</h3>

Structures from modelling, structure prediction and crystallography are always not in the right conformationa and energy for MD calculations.
Energy minimization (EM) is an important step to creating a stable system by reducing bad contacts. EM tries to find the closest structure that is at an energy minimum.

In [ ]:
#Create a folder for energy minimization
%cd ..
!mkdir energy_minim
%cd energy_minim

#copy  all topology files, the complex pdb and crd files from the system preparation folder to the current folder
!cp ../system_prep/*.top ../system_prep/complex.pdb ../system_prep/complex.crd . 

In [ ]:
#If you want to skip the energy minimization step, just copy a minimized system from ./system_prep_done
#!cp ../../AmberTools_run_done/energy_minim_done/* .

<font color="darkred">What is a CRD file in Amber system; how does it differ from PDB or other cordinate files?</font>

- **The EM input file** 

This is a file that sets all the conditions of the energy minimization step like the number of steps, algorithm to use etc.

Visualize the **"em.in"** file. <font color="darkred">What does all the parameters mean?</font>

We will perform this energy minimization ("imin=1"), for 5000 steps, starting with steepest descent and switching to conjugate gradient ("ncyc=500"). 

We will apply a restraint force of 100 to all the residues of the protein (1-163) at the alpha carbon. We will also apply restraint to the ligand atom (residue 164) <font color="darkred">Why is this restraint important?</font>

In [ ]:
## Copy the em.in file to the energy_minim folder from the input folder
!cp ../../input_amber/em.in .

In [ ]:
#Visualize the input file and inspect the parameters
%cat em.in

<font color="darkred"> What does the **"ioutfm=1"** and **"ntxo=2"** control? 

What are the advantages of using the NETCD file format over RST or MDCRD format in writong restart and trajectory information?</font>

- **Run the minimization**

In [ ]:
#since we installed ambertools without the GPU support, we only have access to sander, not pmemd.cuda. 
#If you have access to pmemd.cuda, you can use it instead of sander for faster calculations.

import os
import shutil
os.environ["AMBER_EXE"] = shutil.which("sander")
print(os.environ["AMBER_EXE"])

#AMBER_EXE is now our environment variable that points to the amber executable path.

In [ ]:
!echo $AMBER_EXE

In [ ]:
!$AMBER_EXE -O -i em.in -o em.out -p complex.top -c complex.crd -ref complex.crd -r em.rst

<font color="darkred">Why was complex.crd used twice in the minimization step? Think about the restraints we applied.

How many minimization steps are enough?</font>

- **Was our system well minimized**

We will extract the total energy over the course of the minimization, from the **em.out** file (step_energy.dat) and make a plot.

<font color="darkred">Did our system converge? What is a local minima</font>

In [ ]:
!grep -A1 "NSTEP" em.out | grep -v "NSTEP" | awk '{print $1, $2}' > step_energy.dat

#copy the plot_energy.py file to the energy_minim folder from the plot folder
!cp ../../plots_scripts/plot_energy.py .

In [ ]:
#make the plot
from plot_energy import plot_energy
plot_energy("step_energy.dat")

- **Convert the "RST" output file to "PDB" file for visualization.** 

The minimized strucuture is also inspected to ensure stability and that no artifact occured during the minimization.

In [ ]:
!cpptraj -p complex.top -y em.rst -x em.pdb # CPPTRAJ is the central engine for analysing trajectories.
#!ambpdb -p complex.top < em.rst> em.pdb #ambpdb can also work, but needs special care with netcd files

In [ ]:
view = nv.show_file("em.pdb")
view.clear_representations()
view.add_representation("cartoon", selection="protein")
view.add_representation("line", selection="WAT") 
view.add_representation("sphere", selection="ions")
view

<h3 style="text-align: left; color: darkblue; font-weight: bold;">NVT Equilibration (Heating)</h3>

Let us heat the temperature of the system gradually from a low starting point (often near 0 K), up to the target temperature such as 300 K. This process prevents structural shocks by allowing atoms and solvent molecules to adjust smoothly to thermal motion. A thermostat, commonly *Langevin or Berendsen*, controls the rate of heating and ensures stability during the ramp. Positional restraints are often applied to the solute while the solvent equilibrates around it, maintaining structural integrity. Once the desired temperature is reached and stable, the system is ready to proceed into pressure equilibration or production dynamics.

In [ ]:
#Create a folder for nvt equilibration
%cd ..
!mkdir nvt_equil
%cd nvt_equil

#copy  all topology files, and the restart (.rst) files from the energy minimization folder to the current folder
!cp ../energy_minim/*.top ../energy_minim/em.rst .

In [ ]:
#If you want to skip the NVT equilibration step, just copy an equilibrated system from ./nvt_equil_done
#!cp ../../AmberTools_run_done/nvt_equil_done/* .

- **The NVT input file** 

We will run **nvt equilibration (heating)**; imin=0 (since this is dynamics, rather than minimization with default algorithm being **leapfrog**); for 10,000 steps at 0.002 picosecs. Since we are starting the simulation, irest=0 and we will initialize velocities, ig=-1. We will heat the system from 0.0K to 300K.

This is a very short NVT equilibration. Please feel free to edit the input file to increase the number of steps

Since this is a simulation, trajectory files are generated; we will save the trajectory file every 1000 steps (ntwx = 1000); save the checkpoint/restart file every 1000 steps (ntwr=1000). We will also apply restraints to all the residues of protein (ntr=1).

<font color="darkred"> What is the total time for our heating? </font> That is **10000 steps multiplied 0.002ps = 20ps**

<font color="darkred"> What does all the other parameters mean? </font>

In [ ]:
## Copy the nvt.in file to the nvt_equil folder from the input folder
!cp ../../input_amber/nvt.in .

In [ ]:
#Visualize the input file and inspect the parameters
%cat nvt.in

<font color="darkred"> How is non-bonded interactions handled; why did we choose cut=8?</font>

- **Run the NVT equilibration**

In [ ]:
!$AMBER_EXE -O -i nvt.in -o nvt.out -p complex.top -c em.rst -r nvt.rst.nc -ref em.rst -x nvt.nc

- **Was our system heated to the required temperature?**

We will extract the temperature over the course of the heating, from the **nvt.out** file (step_temp.dat) and make a plot.

<font color="darkred">Did our temperature converge to the darket value (300K)? What thermostatt did we use (ntt = 3)?</font>

In [ ]:
!grep "NSTEP" nvt.out | awk '{print $3, $9}' > step_temp.dat

#copy the plot_temp.py file to the nvt_equil folder
!cp ../../plots_scripts/plot_temp.py .

In [ ]:
#make the plot
from plot_temp import plot_temp
plot_temp("step_temp.dat")

<font color="darkred">What would happen if we do not heat up the system first?</font>

<font color="darkred">Why do we need a restart/checkpoint file now?</font>

- **Convert the ".NC" output file to "PDB" file for visualization.** 

The heated strucuture is also inspected to ensure stability and that no artifact occured during the heating.

In [ ]:
!cpptraj -p complex.top -y nvt.rst.nc -x nvt.pdb # CPPTRAJ is the central engine for analysing trajectories.
#!ambpdb -p complex.top < nvt.rst.nc> nvt.pdb #ambpdb can also work, but needs special care with netcd files

Most time, equilibrations are done in multiple phases; over which the restraints are gradually released until there is no more restraint to allow the system relax. This ensure more sustained relaxation over time.

<h3 style="text-align: left; color: darkblue; font-weight: bold;">NPT Equilibration (Pressure Equilibration)</h3>

We will adjust the system to the target pressure, typically 1 atmosphere, under constant pressure conditions. The barostat dynamically scales the simulation box dimensions so that density and pressure stabilize over time. The relaxation parameter (*gamma_ln*) controls how quickly the system responds to fluctuations, ensuring smooth convergence. Often, positional restraints are applied to the solute while the solvent reorganizes to achieve equilibrium. Once the pressure and volume stabilize, the system is ready to transition into production runs with consistent thermodynamic behavior.

In [ ]:
#Create a folder for npt equilibration
%cd ..
!mkdir npt_equil
%cd npt_equil

#copy  all topology files, and the restart (.rst) files from the nvt equilibration folder to the current folder
!cp ../nvt_equil/*.top ../nvt_equil/nvt.rst.nc .

In [ ]:
#If you want to skip the NPT equilibration step, just copy an equilibrated system from ./npt_equil_done
#!cp ../../AmberTools_run_done/npt_equil_done/* .

- **The NPT input file** 

We will run **npt equilibration (pressure equilibration)**; imin=0 (since this is dynamics, rather than minimization with default algorithm being **leapfrog**); for 100,000 steps at 0.002 picosecs. Since we are continuing the simulation, irest=1 and we will use velocities from the last restart files. We will heat the system from 0.0K to 300K.

As during the heating, trajectory files are generated; we will save the trajectory file every 1000 steps (ntwx = 1000); save the checkpoint/restart file every 1000 steps (ntwr=1000). We will also apply restraints to all the residues of protein.

In [ ]:
## Copy the npt.in file to the npt_equil folder from the input folder
!cp ../../input_amber/npt.in .

In [ ]:
#Visualize the input file and inspect the parameters
%cat npt.in

<font color="darkred"> What barostat are we using for the pressure coupling (ntt=3) and why is our non-bonded cut-off increased to 12 </font>

<font color="darkred"> Why do we run the equilibration for 100000 steps?</font>

The equilibration would be run in **three stages:** first with a **restraint force of 10 and then reduced to 3 and finally removed**. This allows the system to gradually relax over time and attain perfect stability before production run.**

- **Run the First NPT equilibration**

In [ ]:
#Rename the npt.in file into npt_1.in
!cp npt.in npt_1.in

In [ ]:
!$AMBER_EXE -O -i npt_1.in -o npt_1.out -p complex.top -c nvt.rst.nc -r npt_1.rst.nc -ref nvt.rst.nc -x npt_1.nc

- **Run the Second NPT equilibration**

In [ ]:
#Generate the npt_2.in file fromnpt_1.in by changing the restraint weight from 10.0 to 3.0
!sed 's/restraint_wt=10.0/restraint_wt=3.0/' npt_1.in > npt_2.in

In [ ]:
!$AMBER_EXE -O -i npt_2.in -o npt_2.out -p complex.top -c npt_1.rst.nc -r npt_2.rst.nc -ref npt_1.rst.nc -x npt_2.nc

- **Run the Third NPT equilibration**

In [ ]:
#Generate the npt_3.in file fromnpt_2.in by turning off the restraint
!sed 's/ntr=1/ntr=0/' npt_2.in > npt_3.in

In [ ]:
!$AMBER_EXE -O -i npt_3.in -o npt_3.out -p complex.top -c npt_2.rst.nc -r npt_3.rst.nc -ref npt_2.rst.nc -x npt_3.nc

- **Was our system equilibrated to the required pressure?**

We will extract the temperature over the course of the heating, from the **npt_3.out** file (step_press.dat) and make a plot.

In [ ]:
!grep "NSTEP" npt_3.out | awk '{print $3, $12}' > step_press.dat

#copy the plot_press.py file to the nvt_equil folder
!cp ../../plots_scripts/plot_press.py .

In [ ]:
#make the plot
from plot_press import plot_press
plot_press("step_press.dat")

- **Convert the ".NC" output file to "PDB" file for visualization.** 

The equilibrated strucuture is also inspected to ensure stability and that no artifact occured during the equilibration.

<font color="darkred">Why was the pressure not very stable as in the case of the temperature?</font>

In [ ]:
!cpptraj -p complex.top -y npt_3.rst.nc -x npt_3.pdb # CPPTRAJ is the central engine for analysing trajectories.
#!ambpdb -p complex.top < npt_3.rst.nc> npt_3.pdb #ambpdb can also work, but needs special care with netcd files

<h3 style="text-align: left; color: darkblue; font-weight: bold;">MD Run (Production Run)</h3>

A **production run** in molecular dynamics is the main stage where the system is allowed to evolve freely under the chosen ensemble, usually NPT or NVT, **without restraints**. At this point, the system has already been minimized, heated, and equilibrated, so the production run generates the actual trajectory data used for analysis. The simulation is typically much longer than equilibration, often tens to hundreds of nanoseconds, to capture meaningful dynamics. Thermostats and barostats maintain stable temperature and pressure, while the atoms move according to the force field. The resulting trajectory provides insights into structural changes, interactions, and properties that can be compared with experimental data.

Usually, production runs are run in the NPT ensemble (thats continuation from the NPT equilibration)

In [ ]:
#Create a folder for npt equilibration
%cd ..
!mkdir mdrun
%cd mdrun

#copy  all topology files, and the restart (.rst) files from the npt equilibration folder to the current folder
!cp ../npt_equil/*.top ../npt_equil/npt_3.rst.nc .

In [ ]:
#If you want to skip the production step, just copy the run files from ./npt_equil_done
#!cp ../../AmberTools_run_done/mdrun_done/* .

- **The RUN input file** 

Since we are extending the NPT equilibration, the input parameters are same or similar, with only the nstlim exended to account for longer simulation time.
We will keep it at 5,000,000 steps, with no restraints. **This can also be loweed depending on your compute resources**. If you run on CPU only SANDER; then please reduce.

Run the production in four steps by dividing the steps into four 1,250,000 steps. The files are names md_1.in md_2.in md_3.in md_4.in. This division is just one of convenience and will be useful during the trajectory analysis.

In [ ]:
## Copy the md.in file to the mdrun folder from the input folder
!cp ../../input_amber/md_*.in .

In [ ]:
#Visualize the input file and inspect the parameters
%cat md_1.in

For efficiency, the production run should be run over GPUs or MPI processes of the PMEMD CUDA.

Run the commands one after the other, using the restart of the previous run for the next run.

In [ ]:
!$AMBER_EXE -O -i md_1.in -o md.out -p complex.top -c npt_3.rst.nc -r md_1.rst.nc -x traj_1.nc
!$AMBER_EXE -O -i md_2.in -o md.out -p complex.top -c md_1.rst.nc -r md_2.rst.nc -x traj_2.nc
!$AMBER_EXE -O -i md_3.in -o md.out -p complex.top -c md_2.rst.nc -r md_3.rst.nc -x traj_3.nc
!$AMBER_EXE -O -i md_4.in -o md.out -p complex.top -c md_3.rst.nc -r md_4.rst.nc -x traj_4.nc

Frames are also called snapshots and can be calculated from the nslim and ntwx

**Number of frames = nstlim/ntwx = 5,000,000/500 = 10,000 frames in total**

<font color="darkred">Why was the *-ref* flag removed in the command above?</font>

<font color="darkred">What is the most important output for the MD run?</font>

<h3 style="text-align: left; color: darkblue; font-weight: bold;">Summary and Reflections</h3>

The preparation stage governs everything; a good run with a bad system is still a bad run. So, ensure your protein and ligands have the right configuration, atomtypes etc for efficient run. A danger during heating and equilibration is system blowup and breakdown in the periodic condition. For this, adjust the thermostats and barostats until you get what is suitable for your run.

<font color="darkred">What are the most common challenges in setup and running simulations in AmberTools?</font>

<h3 style="text-align: left; color: darkblue; font-weight: bold;">Citations</h3>

- https://ambermd.org/tutorials/
- https://ambermd.org/tutorials/Relaxation.php
- https://ambermd.org/doc12/Amber21.pdf#page=217
- https://ambermd.org/antechamber/example.html
- David A. Case, Hasan Metin Aktulga, Kellon Belfon, David S. Cerutti, et. al; AmberTools. J. Chem. Inf. Model. 23 October 2023; 63 (20): 6183–6191. https://doi.org/10.1021/acs.jcim.3c01153